#reconfiguración general de columnas sin título
Una primera intervención con las bases de datos sería agregarles título de columna para facilitar su lectura y para garantizar que no se confundirán con otras bases cuando se avance con la fusión.
Este primer script será para:

_ Configurar los parámetros de la salida si no estuviese hecho ya.

_ Colocar los encabezados de columnas

_ Generar nombres genéricos para las columnas de EMG (ch1, ch2)

_ Crear vectores de tiempo, de ser necesarios

_ Diseño y aplicación de filtro Notch si correspondiese

_ Visualización comparativa de lo realizado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# 1. Configuración de parámetros (basado en el protocolo de Unicamp)
emg_file = '../data/raw/emg_S001_111.csv' # Asegúrate de que esta ruta coincida con tu carpeta
fs_emg = 2052.0  # Frecuencia de muestreo en Hz
f0 = 60.0        # Frecuencia de red eléctrica en Brasil a filtrar (Notch)
Q = 30.0         # Factor de calidad del filtro

# 2. Carga de datos EMG (solucionando el problema de los encabezados)
# Usamos header=None porque el CSV crudo no trae nombres de columnas
df_emg = pd.read_csv(emg_file, header=None)

# Generar nombres de columnas genéricos (CH_1 a CH_128)
df_emg.columns = [f'CH_{i+1}' for i in range(df_emg.shape[1])]

# Crear vector de tiempo en segundos
df_emg['Time_s'] = np.arange(len(df_emg)) / fs_emg

# 3. Diseño y aplicación del Filtro Notch
b, a = signal.iirnotch(w0=f0, Q=Q, fs=fs_emg)
df_emg['CH_1_Filtered'] = signal.filtfilt(b, a, df_emg['CH_1'])

# 4. Visualización Comparativa
plt.figure(figsize=(15, 6))
plt.plot(df_emg['Time_s'], df_emg['CH_1'], label='Señal Cruda (CH_1)', alpha=0.5, color='gray')
plt.plot(df_emg['Time_s'], df_emg['CH_1_Filtered'], label='Señal Filtrada (Notch 60Hz)', color='blue')
plt.title('Exploración EMG: Impacto del Filtro Notch (60 Hz)')
plt.xlabel('Tiempo (segundos)')
plt.ylabel('Amplitud')
plt.xlim(0, 5) # Hacemos zoom en los primeros 5 segundos
plt.legend()
plt.grid(True)
plt.show()